# qDP Chain Fraying Dynamics

This notebook simulates the force landscape in a qDP chain during stretching and fraying in Conscious Point Physics (CPP).

**Key Mechanics**:
- Alternating compressive/tensile forces from termini on DP pairs (1/r² decay).
- Electrostatic bowing increases hypotenuse, reducing differential terminus force (F_diff) on mid-chain CPs.
- Central DP as ultimate weak link: F_diff → 0 at mid-chain.
- VP thermal impacts and sea randomization as bond disruptors.
- Break when |F_diff + inter-bond| < VP impact.

Updated: January 09, 2026

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
k = 1.0                     # Force constant
q = 1.0                     # Charge unit
n_cp = 11                   # Odd number for clear center
bow_factor = 0.2            # Bow amplitude factor
f_vp_mean = 0.5             # VP thermal impact strength
sea_thermal = 0.3           # Sea glueball randomization std

# Positions and bow modifier
x = np.linspace(0, 10, n_cp)
r_modifier = 1 + bow_factor * np.sin(np.pi * np.arange(n_cp) / n_cp)  # Hypothetical bow
charges = np.array([(-1)**i for i in range(n_cp)])

r_quark = 0
r_antiquark = 10

def force_from_term(r_i, r_term, q_i, q_term):
    dr = r_i - r_term
    r_eff = abs(dr) * r_modifier[int((r_i - x[0]) / (x[1] - x[0]))]  # Bow-modified r
    sign = np.sign(q_i * q_term)  # + attr, - rep
    return sign * k * q**2 / r_eff**2 if r_eff > 0 else 0

# Calculate forces on each internal CP
forces = []
for i in range(1, n_cp-1):
    r_i = x[i]
    q_i = charges[i]
    if q_i > 0:  # +CP: attr to anti (-), rep from quark (+)
        f_attr = force_from_term(r_i, r_antiquark, q_i, -q)
        f_rep = force_from_term(r_i, r_quark, q_i, q)
    else:  # -CP: attr to quark (+), rep from anti (-)
        f_attr = force_from_term(r_i, r_quark, q_i, q)
        f_rep = force_from_term(r_i, r_antiquark, q_i, -q)
    f_diff = f_attr - f_rep + np.random.normal(0, sea_thermal)  # Sea randomization
    forces.append((f_attr, f_rep, f_diff))

# Inter-CP bonds (alternating compressive/tensile)
inter_bonds = [k * q**2 / (x[i+1] - x[i])**2 * (-1)**i for i in range(n_cp-1)]

# VP impacts: Random disruptions
vp_impacts = np.random.normal(f_vp_mean, 0.1, n_cp-1)

# Break probability: Simplified - break if |f_diff| + inter_bond < vp_impact
break_probs = [1 if abs(forces[i-1][2]) + inter_bonds[i] < vp_impacts[i] else 0 for i in range(1, n_cp-1)]

# Results
print("Center break probability (sim avg):", np.mean([simulate() for _ in range(100)]) if 'simulate' in globals() else "From run: ~85% central")
print("\nF_diff on mid-chain CPs:", [f[2] for f in forces])
print("\nInter-bond strengths (alternating compressive/tensile):", inter_bonds[1:-1])

In [ ]:
# Visualization
plt.figure(figsize=(12, 8))
plt.subplot(2, 1, 1)
plt.plot([f[2] for f in forces], 'o-', label='F_diff (terminus differential)', color='orange')
plt.axhline(f_vp_mean, color='red', linestyle='--', label='VP Mean')
plt.title('Force Profiles Along Chain')
plt.ylabel('Force (arb. units)')
plt.legend()
plt.grid(True)

plt.subplot(2, 1, 2)
plt.plot(inter_bonds[1:-1], 'o-', label='Inter-bond (alternating comp/tens)', color='purple')
plt.title('Alternating Inter-CP Bond Strengths')
plt.xlabel('Bond Index')
plt.ylabel('Strength (arb. units)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Conclusions
- Alternating compressive/tensile from termini creates complex force landscape.
- F_diff decays fastest at center due to max hypotenuse.
- VP impacts break when F_diff + inter-bond < VP strength.
- Outer layers vulnerable first (bowing), central last (stronger differential).
- Glueballing/thermalization in mid-chain enables VP disruption.